# Prediction intervals with MMF
This notebook shows how to get calibrated prediction intervals from MMF, using daily sales of a sample of [M5 (Walmart)](https://www.sciencedirect.com/science/article/pii/S0169207021001874) items.

MMF calibrates each model's intervals on its own backtest, so that they reach the requested coverage:
- models with native intervals (here `AutoETS` and `SeasonalNaive`) get their native bands corrected by conformalized quantile regression (CQR);
- the other models (here `CrostonSBA`) get conformal intervals from their backtest errors.

See the [Prediction Intervals](https://github.com/databricks-industry-solutions/many-model-forecasting#prediction-intervals) section of the README for the settings.

### Cluster setup

Use a CPU cluster with [Databricks Runtime 18 for ML](https://docs.databricks.com/aws/en/release-notes/runtime/18ml), or serverless compute. Calibration and scoring run in Spark, so the cluster size only needs to fit the backtest of the local models.

### Install and import packages

In [ ]:
%pip install -r ../../requirements-local.txt --quiet
%pip install datasetsforecast==0.0.8 matplotlib --quiet
dbutils.library.restartPython()

In [ ]:
import numpy as np
import pandas as pd
from datasetsforecast.m5 import M5
from mmf_sa import run_forecast

### Prepare data
We download M5 and keep a random sample of item x store series. The full long table does not fit in a small driver, so we sample the wide CSV before melting it. Leading zeros (before an item's launch) are dropped.

In [ ]:
catalog = "mmf"  # Name of the catalog we use to manage our assets
db = "m5"  # Name of the schema we use to manage our assets (e.g. datasets)
user = spark.sql("select current_user() as user").collect()[0]["user"]
n = 500  # Number of series

_ = spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
_ = spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{db}")

In [ ]:
M5.download(directory="/tmp/m5")
path = "/tmp/m5/m5/datasets"
calendar = pd.read_csv(f"{path}/calendar.csv", usecols=["date"], parse_dates=["date"])
calendar["d"] = [f"d_{i + 1}" for i in range(len(calendar))]
wide = pd.read_csv(f"{path}/sales_train_evaluation.csv")
wide["unique_id"] = wide["item_id"] + "_" + wide["store_id"]
wide = wide.sample(n=n, random_state=7)
day_cols = [c for c in wide.columns if c.startswith("d_")]
train = wide.melt(id_vars=["unique_id"], value_vars=day_cols, var_name="d", value_name="y")
train = train.merge(calendar, on="d").rename(columns={"date": "ds"}).drop(columns="d")
train["y"] = train["y"].astype(float)
train = train.sort_values(["unique_id", "ds"])
train = train[train.groupby("unique_id")["y"].cumsum() > 0]  # drop leading zeros

(
    spark.createDataFrame(train)
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"{catalog}.{db}.daily_train_pi")
)
print(f"{train['unique_id'].nunique()} series, {train['ds'].min().date()} to {train['ds'].max().date()}")

### Run MMF with prediction intervals
`prediction_intervals` is the only new argument. We ask for 80% and 95% intervals, both as a quantile map and as bound columns (`output_format: both`).

The backtest has 3 windows of 28 days (`backtest_length=84`, `stride=28`), which is enough with the default settings: corrections are pooled per ABC x XYZ class of similar series and shrunk toward each series' own errors.

In [ ]:
active_models = [
    "StatsForecastBaselineSeasonalNaive",
    "StatsForecastAutoETS",
    "StatsForecastCrostonSBA",
]

run_id = run_forecast(
    spark=spark,
    train_data=f"{catalog}.{db}.daily_train_pi",
    scoring_data=f"{catalog}.{db}.daily_train_pi",
    scoring_output=f"{catalog}.{db}.daily_pi_scoring_output",
    evaluation_output=f"{catalog}.{db}.daily_pi_evaluation_output",
    group_id="unique_id",
    date_col="ds",
    target="y",
    freq="D",
    prediction_length=28,
    backtest_length=84,
    stride=28,
    metric="smape",
    train_predict_ratio=1,
    active_models=active_models,
    experiment_path=f"/Users/{user}/mmf/m5_prediction_intervals",
    use_case_name="m5_daily_pi",
    prediction_intervals={"enabled": True, "levels": [80, 95], "output_format": "both"},
)

### Forecasts with intervals
`scoring_output` now has `y_quantiles` (a map from quantile level to an array over the horizon) and the bound columns `y_lo_80`, `y_hi_80`, `y_lo_95`, `y_hi_95`.

In [ ]:
display(spark.sql(f"""
    SELECT unique_id, model, ds, y, y_lo_80, y_hi_80, y_lo_95, y_hi_95
    FROM {catalog}.{db}.daily_pi_scoring_output
    WHERE run_id = '{run_id}'
    ORDER BY unique_id, model
    LIMIT 9
"""))

In [ ]:
import matplotlib.pyplot as plt

series = spark.sql(f"""
    SELECT unique_id FROM {catalog}.{db}.daily_pi_scoring_output WHERE run_id = '{run_id}'
    ORDER BY unique_id LIMIT 1
""").collect()[0][0]
history = train[train["unique_id"] == series].tail(120)
fc = spark.sql(f"""
    SELECT * FROM {catalog}.{db}.daily_pi_scoring_output
    WHERE run_id = '{run_id}' AND unique_id = '{series}' AND model = 'StatsForecastAutoETS'
""").toPandas().iloc[0]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(history["ds"], history["y"], color="black", label="history")
ax.plot(fc["ds"], fc["y"], label="AutoETS forecast")
ax.fill_between(fc["ds"], fc["y_lo_95"], fc["y_hi_95"], alpha=0.2, label="95%")
ax.fill_between(fc["ds"], fc["y_lo_80"], fc["y_hi_80"], alpha=0.35, label="80%")
ax.set_title(series)
ax.legend()
plt.show()

### Are the intervals calibrated?
The interval metrics are computed on the backtest **out of sample**: each window is scored with intervals calibrated on the other windows. `coverage_80` should be close to 0.80 and `coverage_95` to 0.95. `msis` is the interval score scaled by each series' seasonal-naive error (lower is better at equal coverage), and the `native_*` rows are the uncorrected native bands of `AutoETS` and `SeasonalNaive`.

In [ ]:
display(spark.sql(f"""
    SELECT model, metric_name, ROUND(AVG(metric_value), 3) AS value
    FROM {catalog}.{db}.daily_pi_evaluation_output_interval_metrics
    WHERE run_id = '{run_id}' AND metric_name RLIKE '^(native_)?(coverage|msis)_'
    GROUP BY model, metric_name
    ORDER BY model, metric_name
"""))

Coverage per ABC x XYZ class. ABC ranks series by volume (A = top 80% of sales), XYZ by volatility (X = most stable). These are the classes MMF pools the calibration over, so each class should be near nominal; small classes are noisier.

In [ ]:
display(spark.sql(f"""
    WITH cls AS (
      SELECT DISTINCT model, unique_id, CONCAT(abc_class, xyz_class) AS abc_xyz
      FROM {catalog}.{db}.daily_pi_evaluation_output_calibration
      WHERE run_id = '{run_id}'
    )
    SELECT m.model, c.abc_xyz, m.metric_name, ROUND(AVG(m.metric_value), 3) AS coverage,
           COUNT(DISTINCT m.unique_id) AS series
    FROM {catalog}.{db}.daily_pi_evaluation_output_interval_metrics AS m
    JOIN cls AS c ON m.model = c.model AND m.unique_id = c.unique_id
    WHERE m.run_id = '{run_id}' AND m.metric_name LIKE 'coverage_%'
    GROUP BY m.model, c.abc_xyz, m.metric_name
    ORDER BY m.model, m.metric_name, c.abc_xyz
"""))

### How each model was calibrated
`calibration_output` holds the corrections per model, series, horizon step and level, with the method MMF used.

In [ ]:
display(spark.sql(f"""
    SELECT model, method, score, coverage_target, level, COUNT(DISTINCT unique_id) AS series,
           ROUND(AVG(q_lo), 3) AS avg_q_lo, ROUND(AVG(q_hi), 3) AS avg_q_hi, FIRST(shrinkage_weight) AS shrinkage_weight
    FROM {catalog}.{db}.daily_pi_evaluation_output_calibration
    WHERE run_id = '{run_id}'
    GROUP BY model, method, score, coverage_target, level
    ORDER BY model, level
"""))

### Delete tables

In [ ]:
#display(spark.sql(f"DROP TABLE IF EXISTS {catalog}.{db}.daily_pi_scoring_output"))
#display(spark.sql(f"DROP TABLE IF EXISTS {catalog}.{db}.daily_pi_evaluation_output"))
#display(spark.sql(f"DROP TABLE IF EXISTS {catalog}.{db}.daily_pi_evaluation_output_calibration"))
#display(spark.sql(f"DROP TABLE IF EXISTS {catalog}.{db}.daily_pi_evaluation_output_interval_metrics"))